In [20]:
import pandas as pd

df = pd.read_csv(
    "/content/clean_jobs_phase2.csv",
    engine="python",
    on_bad_lines="skip"
)

df.shape


(8261, 21)

In [21]:
salary_df = df[df["sal_high"].notna()].copy()

salary_df.shape


(2135, 21)

In [22]:
X = salary_df[
    ["months_experience", "education", "location", "company"]
]

y = salary_df["sal_high"]


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape


((1708, 4), (427, 4))

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

num_features = ["months_experience"]
cat_features = ["education", "location", "company"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipe = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", model)
    ]
)


In [25]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['months_experience']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['education', 'location',
                                                   'company'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [26]:
y_pred = pipe.predict(X_test)

In [27]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

rmse, r2

(np.float64(28738.48524405108), 0.5160832382535779)

In [28]:
# get trained forest from pipeline
rf_model = pipe.named_steps["model"]

# get feature names after encoding
ohe = pipe.named_steps["prep"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(cat_features)

feature_names = num_features + list(cat_names)

importances = rf_model.feature_importances_

feat_imp = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values(by="importance", ascending=False)
    .head(15)
)

feat_imp


,feature,importance
78,"location_Cupertino, CA",0.136296
274,"location_San Francisco, CA",0.063091
995,company_maven,0.063084
0,months_experience,0.043855
881,company_Tata Consultancy Services,0.036858
777,company_Quantitative Systems,0.025279
205,"location_Mountain View, CA",0.016364
122,location_Greater Houston,0.013423
536,company_Facebook,0.012977
212,"location_New York, NY",0.012252


In [29]:
TOP_SKILLS = [
    "python", "java", "sql", "excel", "aws",
    "machine learning", "c++", "javascript",
    "kubernetes", "azure"
]

import ast

def parse_skill_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x

salary_df["skills"] = salary_df["skills"].apply(parse_skill_list)

# 🔧 FIX None values
salary_df["skills"] = salary_df["skills"].apply(
    lambda x: x if isinstance(x, list) else []
)

for skill in TOP_SKILLS:
    salary_df[f"skill_{skill.replace(' ', '_')}"] = salary_df["skills"].apply(
        lambda s: int(skill in s)
    )


In [30]:
skill_cols = [c for c in salary_df.columns if c.startswith("skill_")]

X2 = salary_df[
    ["months_experience", "education", "location", "company"] + skill_cols
]

y2 = salary_df["sal_high"]


In [31]:
from sklearn.model_selection import train_test_split

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2,
    test_size=0.2,
    random_state=42
)


In [32]:
num_features2 = ["months_experience"] + skill_cols
cat_features2 = ["education", "location", "company"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor2 = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features2),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features2)
    ]
)


In [33]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

model2 = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

pipe2 = Pipeline(
    steps=[
        ("prep", preprocessor2),
        ("model", model2)
    ]
)


In [34]:
pipe2.fit(X_train2, y_train2)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['months_experience',
                                                   'skill_python', 'skill_java',
                                                   'skill_sql', 'skill_excel',
                                                   'skill_aws',
                                                   'skill_machine_learning',
                                                   'skill_c++',
                                                   'skill_javascript',
                                                   'skill_kubernetes',
                                                   'skill_azure']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['education', 'location',
                                                   'company'])])),
                ('model',
                 RandomForestRegressor(n_estimators=300, n_jobs=-1,
                                       random_state=42))])

In [35]:
y_pred2 = pipe2.predict(X_test2)

from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rmse2 = np.sqrt(mean_squared_error(y_test2, y_pred2))
r22 = r2_score(y_test2, y_pred2)

rmse2, r22


(np.float64(28714.160688790067), 0.5169020760133693)

In [36]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor2_dense = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features2),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features2)
    ]
)



In [37]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

gb_model = HistGradientBoostingRegressor(
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)

pipe_gb = Pipeline(
    steps=[
        ("prep", preprocessor2_dense),
        ("model", gb_model)
    ]
)


In [38]:
pipe_gb.fit(X_train2, y_train2)


Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['months_experience',
                                                   'skill_python', 'skill_java',
                                                   'skill_sql', 'skill_excel',
                                                   'skill_aws',
                                                   'skill_machine_learning',
                                                   'skill_c++',
                                                   'skill_javascript',
                                                   'skill_kubernetes',
                                                   'skill_azure']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['education', 'location',
                                                   'company'])])),
                ('model',
                 HistGradientBoostingRegressor(learning_rate=0.05, max_depth=8,
                                               random_state=42))])

In [39]:
y_pred_gb = pipe_gb.predict(X_test2)

rmse_gb = np.sqrt(mean_squared_error(y_test2, y_pred_gb))
r2_gb = r2_score(y_test2, y_pred_gb)

rmse_gb, r2_gb


(np.float64(31329.851340284644), 0.4248785220352169)

In [40]:
globals().keys()


dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'files', '_i2', 'TOP_SKILLS', 'ast', 'parse_skill_list', '_i3', '_i4', '_i5', 'pd', '_i6', 'df', '_6', '_i7', 'salary_df', '_7', '_i8', 'X', 'y', '_i9', 'train_test_split', 'X_train', 'X_test', 'y_train', 'y_test', '_9', '_i10', 'ColumnTransformer', 'OneHotEncoder', 'Pipeline', 'RandomForestRegressor', 'num_features', 'cat_features', 'preprocessor', 'model', 'pipe', '_i11', '_11', '_i12', 'y_pred', '_i13', 'mean_squared_error', 'r2_score', 'np', 'rmse', 'r2', '_13', '_i14', 'rf_model', 'ohe', 'cat_names', 'feature_names', 'importances', 'feat_imp', '_14', '_i15', 'skill', '_i16', '_i17', '_i18', 'skill_cols', 'X2', 'y2', '_i19', 'X_train2', 'X_test2', 'y_train2', 'y_test2', '_i20', '_20', '_i21', '_21', '_i22', '_i23', '_23', '_i24', '_i25', '_25', '_i26', '_i27', '_27', '

In [41]:
import joblib
joblib.dump(pipe2, "/content/salary_model.pkl")


['/content/salary_model.pkl']

In [42]:
from google.colab import files
files.download("/content/salary_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>